# Libaries

In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [10]:
import pandas as pd

from src.benchmark import (
    ElasticityConfig,
    DesignMatrixBuilder,
    LogLogElasticityModel,
    ElasticityPipeline,
)

from src.dominick import DominickDataLoader

# Loader

In [11]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")

Dataset shape: (463722, 30)
Columnas: ['store_code', 'upc_code', 'category_code', 'week_id', 'log_liters_sold', 'log_price_per_liter', 'on_promo', 'week_rank', 'sin_52', 'cos_52', 'sin_26', 'cos_26', 'sin_13', 'cos_13', 'weeks_since_first_seen_upc', 'weeks_since_first_seen_store_upc', 'liters_per_upc', 'lag_1_log_liters_sold', 'lag_2_log_liters_sold', 'lag_4_log_liters_sold', 'rolling_mean_4_log_liters_sold', 'rolling_mean_8_log_liters_sold', 'rolling_mean_13_log_liters_sold', 'miss_lag_1', 'miss_lag_2', 'miss_lag_4', 'miss_roll_4', 'miss_roll_8', 'miss_roll_13', 'promo_intensity_store_week']


# Config

In [12]:
config = ElasticityConfig(csv_path="elasticity_dataset.csv")
config

ElasticityConfig(csv_path='elasticity_dataset.csv', target_col='log_liters_sold', price_col='log_price_per_liter', control_cols=['on_promo', 'sin_52', 'cos_52', 'sin_26', 'cos_26', 'promo_intensity_store_week'])

# Pipeline

In [13]:
# Cambia estos nombres si tu dataset usa otros:
group_cols = ["store_code", "upc_code"]

full_df = df.copy()
results = []

for keys, group_df in full_df.groupby(group_cols):
    # Desempaquetar claves (para 2 columnas)
    store_id, upc = keys

    # Opcional: filtrar grupos pequeños
    if len(group_df) < 10:
        continue

    matrix_builder = DesignMatrixBuilder(config)
    model = LogLogElasticityModel(price_col=config.price_col)
    pipeline = ElasticityPipeline(
        matrix_builder=matrix_builder,
        model=model,
    )

    try:
        out = pipeline.run(df=group_df)
        beta = out["elasticity"]
        se = pipeline.model.result.bse[config.price_col]
        ci_low, ci_high = pipeline.model.result.conf_int().loc[config.price_col]

        results.append({
            "store_code": store_id,
            "upc_code": upc,
            "elasticity": beta,
            "elasticity_se": se,
            "elasticity_ci_low": ci_low,
            "elasticity_ci_high": ci_high,
            "r_squared": out["metrics"]["r_squared"],
            "adj_r_squared": out["metrics"]["adj_r_squared"],
            "n_obs": out["metrics"]["n_obs"],
        })
    except Exception as e:
        results.append({
            "store_code": store_id,
            "upc_code": upc,
            "elasticity": None,
            "elasticity_se": None,
            "elasticity_ci_low": None,
            "elasticity_ci_high": None,
            "r_squared": None,
            "adj_r_squared": None,
            "n_obs": len(group_df),
            "error": str(e),
        })


elasticities_df = pd.DataFrame(results)

/home/thebigmonster/Github/.venv/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=16 observations were given.
  return hypotest_fun_in(*args, **kwds)
/home/thebigmonster/Github/.venv/lib/python3.10/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=11 observations were given.
  return hypotest_fun_in(*args, **kwds)


# Save

In [15]:
elasticities_df.to_csv("../data/benchmark_elasticities_store_upc.csv", index=False)
elasticities_df.describe()

,store_code,upc_code,elasticity,elasticity_se,elasticity_ci_low,elasticity_ci_high,r_squared,adj_r_squared,n_obs
count,2656.000000,2.656000e+03,2656.000000,2656.000000,2656.000000,2656.000000,2656.000000,2656.000000,2656.000000
mean,88.995482,4.612043e+09,-3.275647,1.536956,-6.322881,-0.228413,0.349326,0.315905,174.594127
std,35.285331,3.531718e+09,5.601389,5.682517,12.266379,12.926720,0.163405,0.172268,62.402745
min,5.000000,1.820000e+09,-104.894633,0.024585,-230.639661,-23.032169,0.017604,-0.083420,11.000000
25%,72.000000,3.410002e+09,-5.086816,0.576248,-7.058352,-3.366085,0.223168,0.181825,127.000000
50%,98.000000,3.410051e+09,-3.424558,0.780807,-5.121490,-1.675554,0.343466,0.310771,188.000000
75%,116.000000,7.204001e+09,-1.420387,1.174377,-3.277418,0.549992,0.470442,0.444261,216.000000
max,139.000000,7.970963e+10,103.894022,104.576743,8.635803,244.060481,0.828647,0.823145,302.000000
